# FIAP Threat Model AI

## Treinamento do modelo YOLOv8

Este notebook realiza o treinamento do modelo YOLOv8 para detecção de componentes em diagramas de arquitetura de software.

Etapas:

1. Verificar disponibilidade da GPU
2. Instalar dependências
3. Carregar o dataset
4. Validar a estrutura do dataset
5. Treinar o modelo
6. Avaliar os resultados
7. Baixar o modelo treinado

In [1]:
!nvidia-smi

Tue Jul 14 14:24:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Verificação da GPU

O treinamento será realizado utilizando uma GPU Tesla T4 disponibilizada pelo Google Colab.

A utilização de GPU reduz significativamente o tempo de treinamento do modelo.

In [2]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 75.6 MB/s eta 0:00:00


## Instalação das dependências

Nesta etapa é instalada a biblioteca Ultralytics, responsável pelo treinamento do modelo YOLOv8.

In [3]:
from ultralytics import YOLO
from google.colab import files

print("Ultralytics instalado com sucesso!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics instalado com sucesso!


## Upload do Dataset

O dataset utilizado foi gerado pelo projeto **FIAP Dataset Generator**.

Ele contém:

- imagens PNG
- ícones AWC
- ícone Azure
- labels no formato YOLO
- arquivos Draw.io
- arquivo dataset.yaml

In [4]:
uploaded = files.upload()

Saving dataset_v5.zip to dataset_v5.zip


In [5]:
!unzip -q dataset_v5.zip -d /content

In [6]:
!ls /content/dataset_v5

dataset.yaml  drawio  images  labels  metadata.csv  preview  README.md


## Validação da Estrutura

Antes do treinamento, verificamos se o dataset foi extraído corretamente e se possui a estrutura esperada pelo YOLO.

In [7]:
!cat /content/dataset_v5/dataset.yaml

path: .
train: images/train
val: images/val

names:
  0: user
  1: internet
  2: firewall
  3: waf
  4: load_balancer
  5: api_gateway
  6: web_server
  7: application_server
  8: microservice
  9: database
  10: cache
  11: queue
  12: object_storage
  13: identity_provider
  14: monitoring


In [9]:
yaml_content = """
path: /content/dataset_v5
train: images/train
val: images/val

names:
  0: user
  1: internet
  2: firewall
  3: waf
  4: load_balancer
  5: api_gateway
  6: web_server
  7: application_server
  8: microservice
  9: database
  10: cache
  11: queue
  12: object_storage
  13: identity_provider
  14: monitoring
"""

with open("/content/dataset_v5/dataset_colab.yaml", "w") as f:
    f.write(yaml_content)

print("dataset_colab.yaml criado com sucesso!")

dataset_colab.yaml criado com sucesso!


In [10]:
!cat /content/dataset_v5/dataset_colab.yaml


path: /content/dataset_v5
train: images/train
val: images/val

names:
  0: user
  1: internet
  2: firewall
  3: waf
  4: load_balancer
  5: api_gateway
  6: web_server
  7: application_server
  8: microservice
  9: database
  10: cache
  11: queue
  12: object_storage
  13: identity_provider
  14: monitoring


## Treinamento Inicial do Modelo

Nesta etapa é realizado um treinamento curto de 5 épocas utilizando o modelo **YOLOv8n**. O objetivo é validar a configuração do ambiente, verificar se o dataset foi carregado corretamente e confirmar que o treinamento ocorre sem erros antes da execução do treinamento completo.

In [11]:
!ls /content

dataset_v5  dataset_v5.zip  sample_data


In [12]:
from google.colab import files

uploaded = files.upload()

Saving best_v2.pt to best_v2.pt


In [13]:
!ls /content

best_v2.pt  dataset_v5	dataset_v5.zip	sample_data


In [15]:
from pathlib import Path

assert Path("/content/best_v2.pt").exists(), "best_v2.pt não encontrado"
assert Path("/content/dataset_v5/dataset_colab.yaml").exists(), "YAML não encontrado"

print("best_v2.pt encontrado")
print("dataset_v5 encontrado")

best_v2.pt encontrado
dataset_v5 encontrado


In [17]:
from ultralytics import YOLO

# Carrega o modelo treinado anteriormente
model = YOLO("/content/best_v2.pt")

results = model.train(
    data="/content/dataset_v5/dataset_colab.yaml",
    epochs=20,
    imgsz=640,
    batch=16,
    workers=2,
    cache=False,
    device=0,
    patience=8,
    project="/content/runs",
    name="fiap_finetuning_v5",
    exist_ok=True
)

Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_v5/dataset_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/best_v2.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fiap_finetuning_v5, nbs=64, nms=False, opset=None, optimize=False, 

In [18]:
!ls -lh /content/runs/fiap_finetuning_v5/weights

total 12M
-rw-r--r-- 1 root root 6.0M Jul 14 15:12 best.pt
-rw-r--r-- 1 root root 6.0M Jul 14 15:12 last.pt


In [20]:
from google.colab import files

files.download("/content/runs/fiap_finetuning_v5/weights/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>